In [ ]:
# Capture camera frames and save them to disk
import cv2
import os
import time
import matplotlib.pyplot as plt

import numpy as np

# Create a directory to save the captured frames
output_dir = "captured_frames"
os.makedirs(output_dir, exist_ok=True)      

# Perform calibration using the captured frames with checkerboard pattern
square_size = 1.0 # Inch
col, row = 9, 5 # Inner corners

# termination criteria
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((row*col,3), np.float32)
objp[:,:2] = np.mgrid[0:col,0:row].T.reshape(-1,2)

def is_blurry(gray, threshold=100):
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return laplacian_var < threshold

In [ ]:
# Open the default camera (0)
cap = cv2.VideoCapture(0)   
if not cap.isOpened():
    print("Error: Could not open camera.")
    exit()

frame_count = 0

# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

try:
    while True:
        time.sleep(2)

        ret, frame = cap.read()
        if not ret:
            print("Error: Could not read frame.")
            break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if is_blurry(gray):
            continue

        # Find the chess board corners
        ret, corners = cv2.findChessboardCorners(gray, (col, row), None)

        # If found, add object points, image points (after refining them)
        if ret == True:
            objpoints.append(objp)

            corners2 = cv2.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
            imgpoints.append(corners2)

            # Draw and save the corners
            cv2.drawChessboardCorners(frame, (col, row), corners2, ret)
            cv2.imwrite(os.path.join(output_dir, f"frame_{frame_count:04d}.jpg"), frame)
    
            frame_count += 1

            print("Captured frame:", frame_count)

        # Exit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
except KeyboardInterrupt:
    print("Interrupted by user.")
finally:    # When everything is done, release the capture and close windows
    cap.release()
    cv2.destroyAllWindows()
    

In [ ]:
from glob import glob

# Display first few captured frames with detected corners
image_files = glob(os.path.join(output_dir, "frame_*.jpg"))
for img_file in image_files[:5]:  # Show the first few frames with detected corners
    img = cv2.imread(img_file)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.title(f"Detected Corners in {os.path.basename(img_file)}")
    plt.axis('off')
    plt.show()

In [ ]:
# Calibrate the camera
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

# Re-projection error
mean_error = 0
for i in range(len(objpoints)):
    imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2)/len(imgpoints2)
    mean_error += error
print("total error: {}".format(mean_error/len(objpoints)))

print("Camera matrix:\n", mtx)
print("Distortion coefficients:\n", dist)

# Save as json
import json
calibration_data = {
    "camera_matrix": mtx.tolist(),
    "distortion_coefficients": dist.tolist(),
}

with open("calibration_data.json", "w") as f:
    json.dump(calibration_data, f, indent=4)

In [ ]:
# Show undistorted image vs original image
import glob
image_files = glob.glob(os.path.join(output_dir, "frame_*.jpg"))
for image_file in image_files[:5]:
    img = cv2.imread(image_file)
    h, w = img.shape[:2]
    newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 0, (w,h))
    
    # Undistort the image
    dst = cv2.undistort(img, mtx, dist, None, newcameramtx)

    # # Crop the image based on the region of interest
    # x, y, w, h = roi
    # dst = dst[y:y+h, x:x+w]
    
    # Display original and undistorted images side by side (larger + higher resolution)
    fig, axes = plt.subplots(2, 1, figsize=(12, 14), dpi=200)
    fig.subplots_adjust(hspace=0.08)

    axes[0].set_title("Original Image")
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].axis('off')

    axes[1].set_title("Undistorted Image")
    axes[1].imshow(cv2.cvtColor(dst, cv2.COLOR_BGR2RGB))
    axes[1].axis('off')

    plt.show()

: 

In [ ]:
# Show live undistorted video feed
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open camera.")
    raise SystemExit   

try:
    while True:
        ret, img = cap.read()
        if not ret:
            print("Error: Could not read frame.")
            break

        h, w = img.shape[:2]
        newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 0, (w,h))
        
        # Undistort the image
        dst = cv2.undistort(img, mtx, dist, None, newcameramtx)

        # Add nxn grid
        n = 6
        for i in range(1, n):
            cv2.line(dst, (0, i * h // n), (w, i * h // n), (0, 255, 0), 1)
            cv2.line(dst, (i * w // n, 0), (i * w // n, h), (0, 255, 0), 1)

        # Display the undistorted video feed
        cv2.imshow('Undistorted Video Feed', dst)

        # Exit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
except KeyboardInterrupt:
    print("Interrupted by user.")
finally:    # When everything is done, release the capture and close windows
    cap.release()
    cv2.destroyAllWindows()
    

In [ ]:
cap.release()
cv2.destroyAllWindows()